In [1]:
%%bash
# Clona o seu repositório
git clone https://github.com/robertgvds/DL-Cardiac-Segmentation-Research.git

Cloning into 'DL-Cardiac-Segmentation-Research'...


In [2]:
import tensorflow as tf
print("GPUs Encontradas pelo TensorFlow:", tf.config.list_physical_devices('GPU'))

2026-04-07 23:19:36.346937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775603976.586328      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775603976.652418      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775603977.159280      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775603977.159340      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775603977.159343      55 computation_placer.cc:177] computation placer alr

GPUs Encontradas pelo TensorFlow: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
%%bash
# 2. Recriar a estrutura e clonar o repositório original fresco
cd /kaggle/working/DL-Cardiac-Segmentation-Research/models/ukbb_cardiac

# 3. Patch do TensorFlow 1.x (Seguro)
find . -name "*.py" -exec sed -i 's/tf.app.flags/tf.compat.v1.app.flags/g' {} +
find . -name "*.py" -exec sed -i 's/import tensorflow as tf/import tensorflow.compat.v1 as tf\ntf.disable_v2_behavior()/g' {} +

# 4. Patch do BUG de forma cirúrgica (Apenas na linha do wall_thickness)
sed -i '/eval_wall_thickness.py/ s/--output_csv/--output_max_csv temp_max.csv --output_csv/' demo_pipeline.py

# 5. O Segredo do PYTHONPATH: Apontar para a pasta PAI (/models)
export PYTHONPATH="/kaggle/working/DL-Cardiac-Segmentation-Research/models:${PYTHONPATH}"
export TF_CPP_MIN_LOG_LEVEL=2

echo "Iniciando o Teste 1 (Baseline Definitivo)..."
python3 demo_pipeline.py

Iniciando o Teste 1 (Baseline Definitivo)...
Start deployment on the data set ...
1
  Reading demo_image/1/sa.nii.gz ...
Start deployment on the data set ...
1
  Reading demo_image/1/la_2ch.nii.gz ...
Start deployment on the data set ...
1
  Reading demo_image/1/la_4ch.nii.gz ...
******************************
  Short-axis image analysis
******************************
Deploying the segmentation network ...
Evaluating ventricular volumes ...
Evaluating myocardial wall thickness ...
******************************
  Long-axis image analysis
******************************
Deploying the segmentation network ...
Evaluating atrial volumes ...
******************************
  Aortic image analysis
******************************
Deploying the segmentation network ...
Evaluating atrial areas ...
Done.


2026-04-07 23:20:38.304890: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775604038.324859     131 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775604038.331377     131 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775604038.350090     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775604038.350146     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775604038.350154     131 computation_placer.cc:177] computation placer alr

In [4]:
%%bash
cd /kaggle/working/DL-Cardiac-Segmentation-Research/models/ukbb_cardiac

echo "1. Consertando o erro do Nibabel (get_data -> get_fdata)..."
find . -name "*.py" -exec sed -i 's/\.get_data()/\.get_fdata()/g' {} +

echo "2. Consertando o grafo do TensorFlow (FusedBatchNormGrad)..."
# Criamos um pequeno script Python que edita os ficheiros deploy_network.py para nós
cat << 'EOF' > patch_tf.py
import os
import re

files_to_patch = [
    "common/deploy_network.py",
    "common/deploy_network_ao.py"
]

patch_lines = [
    "meta_graph_def = tf.compat.v1.MetaGraphDef()",
    "with open('{0}.meta'.format(FLAGS.model_path), 'rb') as f:",
    "    meta_graph_def.ParseFromString(f.read())",
    "for node in meta_graph_def.graph_def.node:",
    "    if node.op == 'FusedBatchNormGrad':",
    "        if '_output_shapes' in node.attr:",
    "            del node.attr['_output_shapes']",
    "saver = tf.compat.v1.train.import_meta_graph(meta_graph_def)"
]

for filepath in files_to_patch:
    if os.path.exists(filepath):
        with open(filepath, 'r') as file:
            content = file.read()
        
        # Encontra a linha onde o modelo é carregado e capta a sua indentação (espaços)
        match = re.search(r'([ \t]*)saver = tf\.(?:compat\.v1\.)?train\.import_meta_graph.*', content)
        if match:
            indent = match.group(1)
            # Monta o nosso bloco de correção com a mesma indentação
            patched_block = '\n'.join([indent + line for line in patch_lines])
            # Substitui a linha antiga pelo nosso bloco
            content = content[:match.start()] + patched_block + content[match.end():]
            
            with open(filepath, 'w') as file:
                file.write(content)
EOF

python3 patch_tf.py

echo "3. Reiniciando o pipeline com a rede corrigida..."
export PYTHONPATH="/kaggle/working/DL-Cardiac-Segmentation-Research/models:${PYTHONPATH}"
export TF_CPP_MIN_LOG_LEVEL=2
python3 demo_pipeline.py

1. Consertando o erro do Nibabel (get_data -> get_fdata)...
2. Consertando o grafo do TensorFlow (FusedBatchNormGrad)...
3. Reiniciando o pipeline com a rede corrigida...
Start deployment on the data set ...
1
  Reading demo_image/1/sa.nii.gz ...
  Segmenting full sequence ...
  Segmentation time = 12.419434s
  ED frame = 0, ES frame = 17
  Saving segmentation ...
2
  Reading demo_image/2/sa.nii.gz ...
  Segmenting full sequence ...
  Segmentation time = 6.805268s
  ED frame = 0, ES frame = 18
  Saving segmentation ...
Average segmentation time = 9.612s per sequence
Including image I/O, CUDA resource allocation, it took 23.968s for processing 2 subjects (11.984s per subjects).
1
2
Start deployment on the data set ...
1
  Reading demo_image/1/la_2ch.nii.gz ...
  Segmenting full sequence ...
  Segmentation time = 3.034551s
  ED frame = 0, ES frame = 21
  Saving segmentation ...
2
  Reading demo_image/2/la_2ch.nii.gz ...
  Segmenting full sequence ...
  Segmentation time = 0.941434s
  ED 

2026-04-07 23:31:22.185381: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775604682.209870     369 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775604682.217970     369 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775604682.237439     369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775604682.237487     369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775604682.237491     369 computation_placer.cc:177] computation placer alr